# FlashVSR-Pro on Google Colab

Select **Runtime → Change runtime type → A100 or L4 GPU**, then run these cells in order. T4/P100 are unsupported. This setup uses Python 3.10–3.12 and CUDA Toolkit 12.4+ within the 12.x series.

The installation and inference run through the notebook's Python executable in separate processes. Start with a short video. Save your output before the runtime expires.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

project = Path('/content/FlashVSR-Pro')
repo_url = 'https://github.com/delvilo/FlashVSR-Pro.git'
branch = 'main'  # Set a feature branch here when testing an unmerged change.

if not (3, 10) <= sys.version_info[:2] <= (3, 12):
    raise RuntimeError('Use a runtime with Python 3.10–3.12 for the pinned dependencies.')
subprocess.run([sys.executable, '-c', """
import torch
if not torch.cuda.is_available():
    raise SystemExit('Select a GPU runtime before continuing.')
major, minor = torch.cuda.get_device_capability()
if major not in (8, 9):
    raise SystemExit('Select A100 or L4. This baseline does not support T4/P100.')
print(torch.cuda.get_device_name(), 'compute capability', major, minor)
"""], check=True)
subprocess.run(['nvcc', '--version'], check=True)


## 1. Install the project

CUDA must already be installed in the runtime. If `nvcc` points to an incompatible version, select an installed CUDA 12.4+ toolkit in the 12.x series using `install_env['CUDA_HOME']` below. The installer validates the selected toolkit before installing packages. Compilation may take several minutes.


In [ ]:
subprocess.run(['apt-get', 'update'], check=True)
subprocess.run(['apt-get', 'install', '-y', 'build-essential', 'git', 'git-lfs', 'ffmpeg', 'python3-dev'], check=True)
if not project.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', branch, repo_url, str(project)], check=True)

install_env = os.environ.copy()
install_env['FLASHVSR_PYTHON'] = sys.executable
# install_env['CUDA_HOME'] = '/usr/local/cuda-12.4'
subprocess.run(['bash', 'scripts/install.sh'], cwd=project, env=install_env, check=True)


## 2. Download model weights

The Hugging Face snapshot contains the DiT, low-quality projector, and decoders. The fixed prompt tensor is already included in the project checkout.


In [ ]:
subprocess.run([
    sys.executable, '-c',
    "from huggingface_hub import snapshot_download; "
    "snapshot_download('JunhaoZhuang/FlashVSR-v1.1', local_dir='models/FlashVSR-v1.1')"
], cwd=project, check=True)


## 3. Choose a video

Use the Files panel to upload a video to `/content`, or mount Google Drive and set its full path. Configure the input and output paths below. The default sample is downloaded and verified on demand. Selecting your own video skips the sample download.


In [ ]:
input_video = project / 'inputs/example0.mp4'  # For example: Path('/content/input.mp4')
output_video = Path('/content/enhanced.mp4')
mode = 'tiny'  # full, tiny, tiny-long
scale = '4.0'

if input_video == project / 'inputs/example0.mp4':
    subprocess.run([
        sys.executable, str(project / 'scripts/download_samples.py'), 'example0.mp4'
    ], cwd=project, check=True)
if not input_video.is_file():
    raise FileNotFoundError(f'Upload a video and set input_video: {input_video}')
output_video.parent.mkdir(parents=True, exist_ok=True)


## 4. Run super-resolution

Tiled inference reduces VRAM usage. For full mode, the cell also enables tiled VAE decoding. Long videos may need smaller segments with `long_video_worker.py`; see the README.


In [ ]:
command = [
    sys.executable, str(project / 'infer.py'),
    '-i', str(input_video), '-o', str(output_video),
    '--mode', mode, '--scale', scale, '--tile-dit', '--keep-audio'
]
if mode == 'full':
    command.append('--tile-vae')
subprocess.run(command, cwd=project, check=True)
if not output_video.is_file():
    raise FileNotFoundError(f'Expected output was not created: {output_video}')
print(f'Output: {output_video}')


## 5. Download the output

For large files, set `output_video` to a path on mounted Google Drive instead. Colab's local files disappear when the runtime is reset.


In [ ]:
from google.colab import files
files.download(str(output_video))
